# Importações

In [1]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import shap

from matplotlib.ticker import FuncFormatter
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor as RF

import pandas as pd

SEED = 100
SEEDS = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
CAMPUS_TREINO = "PARANAGUÁ"
K_FOLDS = 4
N_CLASSES = 10


def reset_seed(rnd_seed=SEED):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)


def calcular_rrmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    mean_y_true = np.mean(y_true)

    rrmse = rmse / mean_y_true
    return rrmse


reset_seed()


/home/eduardo/miniconda3/envs/rapids-24.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar Datasets

In [2]:
df = pd.read_csv("./dados/dados_mesclados.csv", sep=';', decimal='.')
df["DATA"] = pd.to_datetime(df["DATA"], format="%Y-%m-%d")
df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,CURSOS_GRAD_VESPERTINO,CURSOS_GRAD_NOTURNO,CURSOS_POS,FÉRIAS,FERIADO,COVID,GREVE,CAMPUS,REGIÃO,ORDEM
0,11400.0,2015-02-28,18.0,22.0,25.0,27.0,692.0,33.0,4.0,18.0,...,0.0,1.428571,1.0,0,3,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,1
1,18427.0,2015-03-31,12.0,19.0,24.0,26.0,729.0,32.0,2.0,12.0,...,0.0,2.000000,1.0,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,2
2,14274.0,2015-04-30,13.0,19.0,21.0,24.0,640.0,29.0,6.0,13.0,...,0.0,2.000000,1.0,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,3
3,11987.0,2015-05-31,7.0,11.0,18.0,23.0,557.0,28.0,9.0,7.0,...,0.0,2.000000,1.0,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,4
4,9006.0,2015-06-30,4.0,10.0,18.0,22.0,542.0,29.0,1.0,4.0,...,0.0,2.000000,1.0,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2378,5640.0,2024-06-30,-0.0,8.0,17.0,20.0,499.0,26.0,1.0,-0.0,...,1.0,1.000000,0.0,0,0,0,30,UNIÃO DA VITÓRIA,REGIÃO SUL,81
2379,11687.0,2024-07-31,1.0,9.0,13.0,18.0,413.0,25.0,6.0,1.0,...,1.0,1.000000,0.0,12,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,82
2380,11129.0,2024-08-31,-3.0,7.0,15.0,21.0,468.0,31.0,1.0,-3.0,...,1.0,1.000000,0.0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,83
2381,9690.0,2024-09-30,10.0,13.0,20.0,24.0,591.0,34.0,3.0,10.0,...,1.0,1.000000,0.0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,84


# Tratamento dos dados
## Simplificação dos valores decimais

In [3]:
# Arredondamento dos dados, pelas casas decimais serem irrelevantes
df = df.round()

# Converte os dados para o tipo inteiro
df[df.drop(columns=["DATA", "CAMPUS", "REGIÃO"]).columns] = df[
    df.drop(columns=["DATA", "CAMPUS", "REGIÃO"]).columns].astype(int)

df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,CURSOS_GRAD_VESPERTINO,CURSOS_GRAD_NOTURNO,CURSOS_POS,FÉRIAS,FERIADO,COVID,GREVE,CAMPUS,REGIÃO,ORDEM
0,11400,2015-02-28,18,22,25,27,692,33,4,18,...,0,1,1,0,3,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,1
1,18427,2015-03-31,12,19,24,26,729,32,2,12,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,2
2,14274,2015-04-30,13,19,21,24,640,29,6,13,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,3
3,11987,2015-05-31,7,11,18,23,557,28,9,7,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,4
4,9006,2015-06-30,4,10,18,22,542,29,1,4,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2378,5640,2024-06-30,0,8,17,20,499,26,1,0,...,1,1,0,0,0,0,30,UNIÃO DA VITÓRIA,REGIÃO SUL,81
2379,11687,2024-07-31,1,9,13,18,413,25,6,1,...,1,1,0,12,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,82
2380,11129,2024-08-31,-3,7,15,21,468,31,1,-3,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,83
2381,9690,2024-09-30,10,13,20,24,591,34,3,10,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,84


## Remoção dos outliers

In [4]:
dataframes = []

for campus, dados in df.groupby("CAMPUS"):
    q1 = dados["CONSUMO"].quantile(0.25)
    q3 = dados["CONSUMO"].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    dados["CONSUMO"] = np.where(dados["CONSUMO"] < limite_inferior, limite_inferior, dados["CONSUMO"])
    dados["CONSUMO"] = np.where(dados["CONSUMO"] > limite_superior, limite_superior, dados["CONSUMO"])

    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)
df.to_csv("./dados/dados_tratados.csv", index=False, sep=";", decimal=".")

df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,CURSOS_GRAD_VESPERTINO,CURSOS_GRAD_NOTURNO,CURSOS_POS,FÉRIAS,FERIADO,COVID,GREVE,CAMPUS,REGIÃO,ORDEM
0,11400.0,2015-02-28,18,22,25,27,692,33,4,18,...,0,1,1,0,3,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,1
1,18427.0,2015-03-31,12,19,24,26,729,32,2,12,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,2
2,14274.0,2015-04-30,13,19,21,24,640,29,6,13,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,3
3,11987.0,2015-05-31,7,11,18,23,557,28,9,7,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,4
4,9006.0,2015-06-30,4,10,18,22,542,29,1,4,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2378,5640.0,2024-06-30,0,8,17,20,499,26,1,0,...,1,1,0,0,0,0,30,UNIÃO DA VITÓRIA,REGIÃO SUL,81
2379,11687.0,2024-07-31,1,9,13,18,413,25,6,1,...,1,1,0,12,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,82
2380,11129.0,2024-08-31,-3,7,15,21,468,31,1,-3,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,83
2381,9690.0,2024-09-30,10,13,20,24,591,34,3,10,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,84


# Normalização

In [5]:
dataframes = []

for campus, dados in df.groupby("CAMPUS"):
    scaler = MinMaxScaler()
    dados[["CONSUMO"]] = scaler.fit_transform(dados[["CONSUMO"]]).round(
        15)  # Reduz a precisão para evitar problemas de ultrapassagem da escala 1
    dataframes.append(dados)

df_normalizado = pd.concat(dataframes, ignore_index=True)
df_normalizado.to_csv("./dados/dados_normalizados.csv", index=False, sep=";", decimal=".")

df_normalizado

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,CURSOS_GRAD_VESPERTINO,CURSOS_GRAD_NOTURNO,CURSOS_POS,FÉRIAS,FERIADO,COVID,GREVE,CAMPUS,REGIÃO,ORDEM
0,0.502037,2015-02-28,18,22,25,27,692,33,4,18,...,0,1,1,0,3,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,1
1,0.846886,2015-03-31,12,19,24,26,729,32,2,12,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,2
2,0.643078,2015-04-30,13,19,21,24,640,29,6,13,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,3
3,0.530844,2015-05-31,7,11,18,23,557,28,9,7,...,0,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,4
4,0.384551,2015-06-30,4,10,18,22,542,29,1,4,...,0,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2378,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,1,1,0,0,0,0,30,UNIÃO DA VITÓRIA,REGIÃO SUL,81
2379,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,1,1,0,12,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,82
2380,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,83
2381,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,1,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,84


# Criação das Classes


In [6]:
def formater_thousands(x, pos):
    return f'{x:.2f}'.replace('.', ',')


def mapear_classe(consumo, intervalos_quantils):
    for i, (inicio_intervalo, fim_intervalo) in enumerate(intervalos_quantils):
        if inicio_intervalo <= consumo <= fim_intervalo:
            return i


df_classes = df_normalizado.copy()
quantiles = np.linspace(0, 1, N_CLASSES + 1)
intervalos = df_classes["CONSUMO"].quantile(quantiles).to_numpy()

classes = [[intervalos[i], intervalos[i + 1]] for i in range(len(intervalos) - 1)]
df_classes["CLASSE"] = df_classes["CONSUMO"].apply(lambda val: mapear_classe(val, classes)).astype(int)
df_classes.to_csv(f"./dados/{N_CLASSES}_classes_normalizadas.csv", index=False, sep=";", decimal=".")
df_classes

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,CURSOS_GRAD_NOTURNO,CURSOS_POS,FÉRIAS,FERIADO,COVID,GREVE,CAMPUS,REGIÃO,ORDEM,CLASSE
0,0.502037,2015-02-28,18,22,25,27,692,33,4,18,...,1,1,0,3,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,1,6
1,0.846886,2015-03-31,12,19,24,26,729,32,2,12,...,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,2,9
2,0.643078,2015-04-30,13,19,21,24,640,29,6,13,...,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,3,8
3,0.530844,2015-05-31,7,11,18,23,557,28,9,7,...,2,1,0,0,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,4,6
4,0.384551,2015-06-30,4,10,18,22,542,29,1,4,...,2,1,0,1,0,0,ASSIS CHATEAUBRIAND,REGIÃO OESTE,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2378,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,1,0,0,0,0,30,UNIÃO DA VITÓRIA,REGIÃO SUL,81,3
2379,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,1,0,12,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,82,9
2380,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,83,9
2381,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,1,0,0,0,0,0,UNIÃO DA VITÓRIA,REGIÃO SUL,84,9


## Análise das Classes

In [7]:
def plot_classes(dados, campus, plot_serie=True):
    plt.figure(figsize=(18, 5))
    plt.rcParams['xtick.labelsize'] = 15
    plt.rcParams['ytick.labelsize'] = 15
    plt.rcParams.update({'font.size': 15})

    for i, (inicio, fim) in enumerate(classes):
        plt.fill_between(
            dados["DATA"],
            inicio,
            fim,
            alpha=0.65,
            label=f"Classe {i} ({inicio:.5f} - {fim:.5f})"
        )
    if plot_serie:
        plt.plot(dados["DATA"], dados["CONSUMO"], label="Consumo", color="black")

    plt.xlabel('MESES')
    plt.ylabel('CONSUMO NORMALIZADO')

    ax = plt.gca()
    ax.yaxis.set_major_formatter(FuncFormatter(formater_thousands))
    ax.set_facecolor('white')
    plt.grid(True, color='white', linestyle="--", linewidth=0.75)
    plt.legend(facecolor='white', framealpha=0.5, bbox_to_anchor=(1.05, 1))
    plt.savefig(f"./dados/classes/{N_CLASSES} Classes {campus}.png", bbox_inches='tight')
    plt.close()


def plot_distribuicao_classes(dados, campus):
    distribuicao_classes = dados["CLASSE"].value_counts()

    plt.figure(figsize=(12, 6))
    bars = plt.bar(distribuicao_classes.index, distribuicao_classes.values, color="blue", alpha=0.8)

    for bar in bars:
        altura = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, altura / 2, f"{int(altura)}",
                 ha="center", va="center", fontsize=12, color="white")

    plt.xlabel("CLASSE", fontsize=14)
    plt.ylabel("QUANTIDADE DE DADOS", fontsize=14)
    plt.xticks(df_classes["CLASSE"].unique(), fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    plt.savefig(f"./dados/classes/Distribuição {N_CLASSES} Classes {campus}.png", bbox_inches="tight")
    plt.close()


for campus, dados in df_classes.groupby("CAMPUS"):
    plot_classes(dados, campus)
    plot_distribuicao_classes(dados, campus)

plot_classes(df_classes, "_TODOS", False)
plot_distribuicao_classes(df_classes, "_TODOS_")



# Criação dos Lags
## Regressão

In [8]:
dataframes = []

for campus, dados in df_normalizado.sort_values("DATA").groupby("CAMPUS"):
    lags = {f'LAG_{i:02d}': dados['CONSUMO'].shift(i) for i in range(1, 12 + 1)}
    dados = pd.concat([dados, pd.DataFrame(lags)], axis=1)
    dados = dados.dropna()
    dados["ORDEM"] = range(1, len(dados) + 1)
    dataframes.append(dados)

df_normalizado = pd.concat(dataframes, ignore_index=True)
df_normalizado.to_csv("./dados/dados_normalizados_lagados.csv", index=False, sep=";", decimal=".")
df_normalizado

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
0,0.615105,2016-02-29,19,22,25,28,734,33,4,19,...,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886,0.502037
1,0.711047,2016-03-31,12,17,22,28,685,32,3,12,...,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886
2,0.633361,2016-04-30,4,9,23,28,702,33,2,4,...,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078
3,0.406291,2016-05-31,4,11,16,22,490,27,9,4,...,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844
4,0.362467,2016-06-30,-1,7,14,20,408,27,3,-1,...,0.711047,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248,0.743880
2091,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248
2092,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038
2093,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,0.359224,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059


## Classificação

In [9]:
dataframes = []

for campus, dados in df_classes.sort_values("DATA").groupby("CAMPUS"):
    lags = {f'LAG_{i:02d}': dados['CLASSE'].shift(i) for i in range(1, 12 + 1)}
    dados = pd.concat([dados, pd.DataFrame(lags)], axis=1)
    dados = dados.dropna()
    dados[list(lags.keys())] = dados[list(lags.keys())].astype(int)  # Converte os lags para int
    dados["ORDEM"] = range(1, len(dados) + 1)
    dataframes.append(dados)

df_classes = pd.concat(dataframes, ignore_index=True)
df_classes.to_csv(f"./dados/{N_CLASSES}_classes_normalizadas_lagadas.csv", index=False, sep=";", decimal=".")

df_classes

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
0,0.615105,2016-02-29,19,22,25,28,734,33,4,19,...,7,7,4,3,3,4,6,8,9,6
1,0.711047,2016-03-31,12,17,22,28,685,32,3,12,...,7,7,7,4,3,3,4,6,8,9
2,0.633361,2016-04-30,4,9,23,28,702,33,2,4,...,4,7,7,7,4,3,3,4,6,8
3,0.406291,2016-05-31,4,11,16,22,490,27,9,4,...,7,4,7,7,7,4,3,3,4,6
4,0.362467,2016-06-30,-1,7,14,20,408,27,3,-1,...,8,7,4,7,7,7,4,3,3,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,9,9,5,9,9,7,8,9,7,8
2091,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,8,9,9,5,9,9,7,8,9,7
2092,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,3,8,9,9,5,9,9,7,8,9
2093,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,3,3,8,9,9,5,9,9,7,8


# Importância das Features

In [ ]:
shap.initjs()

def calcular_importancia(dados):
    x = dados.drop(columns=["CONSUMO", "DATA", "CAMPUS", "REGIÃO"])
    y = dados["CONSUMO"]

    rf = RF(random_state=SEED)
    rf.fit(x, y)

    explainer_rf = shap.explainers.TreeExplainer(rf)

    shap_rf = explainer_rf(x)
    importancia = pd.DataFrame(list(zip(x.columns, np.abs(shap_rf.values).mean(0))), columns=["FEATURE", "IMPORTÂNCIA"])
    importancia["IMPORTÂNCIA"] = importancia["IMPORTÂNCIA"].round(15)
    importancia = importancia.sort_values(by="FEATURE")
    importancia = importancia.set_index("FEATURE")
    return importancia


# Calcula a importância individual de cada campus
importancias = pd.DataFrame(index=pd.Index(df_normalizado.drop(columns=["CONSUMO", "DATA", "CAMPUS", "REGIÃO"]).columns, name="FEATURE"))
for campus, dados in df_normalizado.groupby("CAMPUS"):
    importancias[[campus]] = calcular_importancia(dados)[["IMPORTÂNCIA"]]

# Calcula a importância geral de todos os campus
importancias[["_TODOS_"]] = calcular_importancia(df_normalizado)[["IMPORTÂNCIA"]]

# Calcula a média das importâncias
importancias["MÉDIA"] = importancias.mean(axis=1)
importancias = importancias.sort_values(by=["MÉDIA"], ascending=False)
importancias.to_csv("./resultados/features/importancia_regressao.csv", index=True, sep=";", decimal=".")

importancias

# Remoção de Features

In [10]:
importancias = pd.read_csv("./resultados/features/importancia_regressao.csv", sep=";", decimal=".", header=0)

features_selecao = importancias.copy()
features_selecao['PERCENTUAL'] = features_selecao['MÉDIA'] / features_selecao['MÉDIA'].sum()

# Remove as features com importância 0
features_selecao = features_selecao[features_selecao["PERCENTUAL"] > 0]

# Seleciona as features com importância percentual menor que 1% para serem sorteadas
features_sorteio = features_selecao[features_selecao["PERCENTUAL"] < 0.01].copy()

features_sorteio["PERCENTUAL"] = 1 - features_sorteio["PERCENTUAL"]
features_sorteio["PROBABILIDADE"] = features_sorteio["PERCENTUAL"] / features_sorteio["PERCENTUAL"].sum()

fitness_regressao = {}

for seed in SEEDS:
    reset_seed(seed)
    for i in range(0, 100):
        print(f"SEED: {seed} - Iter {i}")
        rf = RF(random_state=SEED)

        # Remove features com menor importância
        opt_features = np.random.choice(
            features_sorteio['FEATURE'],
            size=np.random.randint(int(len(features_sorteio)/2), len(features_sorteio)), # Seleciona um número aleatório de features
            replace=False, # Impede a repetição de features na seleção
            p=features_sorteio['PROBABILIDADE']
        ).tolist() + ["CONSUMO", "DATA", "CAMPUS", "REGIÃO"]

        # Treinamento com apenas o campus com a maior quantidade de dados
        dataset_treino = df_normalizado[df_normalizado["CAMPUS"] == CAMPUS_TREINO]
        cvs = []
        for i_treino, i_teste in TimeSeriesSplit(n_splits=K_FOLDS, test_size=1).split(dataset_treino):
            x_treino = df_normalizado.drop(columns=opt_features).iloc[i_treino]
            y_treino = df_normalizado["CONSUMO"].iloc[i_treino]
            # Teste com os dados de todos os campus
            data_teste = dataset_treino.iloc[i_teste]["DATA"].values[0]
            df_teste = df_normalizado.loc[df_normalizado["DATA"] == data_teste]
            x_teste = df_teste.drop(columns=opt_features)
            y_teste = df_teste["CONSUMO"]

            y_previsto = []
            rf.fit(x_treino, y_treino)
            for _, row in x_teste.iterrows():
                previsao = rf.predict(pd.DataFrame([row.values], columns=x_teste.columns))[0]
                y_previsto.append(previsao)
            erros = calcular_rrmse(y_teste, y_previsto)
            cvs.append(erros.mean())

        print("RRMSE:", np.array(cvs).mean())
        fitness_regressao[tuple(dataset_treino.drop(columns=opt_features).columns.to_list())] = np.array(cvs).mean().round(5)

fitness_regressao = pd.DataFrame(list(fitness_regressao.items()), columns=["FEATURES", "RRMSE"])
fitness_regressao.to_csv("./resultados/features/fitness_features_regressao.csv", index=False, sep=";", decimal=".")
fitness_regressao

SEED: 1000 - Iter 0
RRMSE: 0.35920811347261195
SEED: 1000 - Iter 1
RRMSE: 0.3628986321942949
SEED: 1000 - Iter 2
RRMSE: 0.3667823433422729
SEED: 1000 - Iter 3
RRMSE: 0.3651061454139495
SEED: 1000 - Iter 4
RRMSE: 0.3589200505156557
SEED: 1000 - Iter 5
RRMSE: 0.3614157311111169
SEED: 1000 - Iter 6
RRMSE: 0.3612832395487323
SEED: 1000 - Iter 7
RRMSE: 0.3615181981640747
SEED: 1000 - Iter 8
RRMSE: 0.36613843762069254
SEED: 1000 - Iter 9
RRMSE: 0.35862503449491767
SEED: 1000 - Iter 10
RRMSE: 0.3579562422624928
SEED: 1000 - Iter 11
RRMSE: 0.35870138587704326
SEED: 1000 - Iter 12
RRMSE: 0.35445660687395475
SEED: 1000 - Iter 13
RRMSE: 0.3671717652499816
SEED: 1000 - Iter 14
RRMSE: 0.36681644158675963
SEED: 1000 - Iter 15
RRMSE: 0.36131762476322177
SEED: 1000 - Iter 16
RRMSE: 0.36271977636829567
SEED: 1000 - Iter 17
RRMSE: 0.36310350675629277
SEED: 1000 - Iter 18
RRMSE: 0.361148021652012
SEED: 1000 - Iter 19
RRMSE: 0.35649197289602974
SEED: 1000 - Iter 20
RRMSE: 0.3639422019455464
SEED: 1000 - I

,FEATURES,RRMSE
0,"(TEMP_MÉD_ACC_MENS, TEMP_MIN_MAX_MENS, TEMP_MI...",0.35921
1,"(TEMP_MÉD_ACC_MENS, PRECIPITAÇÃO_MÉD_MENS, TEM...",0.36290
2,"(TEMP_MIN_MÉD_MENS, TEMP_MÉD_MIN_MENS, TEMP_MÉ...",0.36678
3,"(TEMP_MIN_MÉD_MENS, TEMP_MÉD_MIN_MENS, TEMP_MÉ...",0.36511
4,"(TEMP_MIN_MÉD_MENS, TEMP_MÉD_MAX_MENS, TEMP_MÉ...",0.35892
...,...,...
986,"(TEMP_MÉD_MÉD_MENS, TEMP_MÉD_ACC_MENS, TEMP_MI...",0.36763
987,"(TEMP_MÉD_MÉD_MENS, TEMP_MÉD_ACC_MENS, TEMP_MI...",0.37153
988,"(TEMP_MÉD_MAX_MENS, TEMP_MÉD_ACC_MENS, PRECIPI...",0.35888
989,"(TEMP_MÉD_ACC_MENS, PRECIPITAÇÃO_MÉD_MENS, TEM...",0.36603


In [11]:
fitness_regressao = pd.read_csv("./resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")
fitness_regressao["RRMSE"] = fitness_regressao["RRMSE"].round(5)
fitness_regressao = fitness_regressao.sort_values("RRMSE")
fitness_regressao


,FEATURES,RRMSE
752,"('TEMP_MÉD_ACC_MENS', 'TEMP_MIN_MAX_MENS', 'TE...",0.35194
405,"('TEMP_MÉD_MAX_MENS', 'TEMP_MÉD_ACC_MENS', 'PR...",0.35226
236,"('TEMP_MÉD_ACC_MENS', 'TEMP_MIN_ACC_MENS', 'TE...",0.35266
824,"('TEMP_MÉD_ACC_MENS', 'TEMP_MIN_ACC_MENS', 'TE...",0.35286
228,"('TEMP_MÉD_MAX_MENS', 'TEMP_MÉD_ACC_MENS', 'TE...",0.35288
...,...,...
726,"('TEMP_MÉD_ACC_MENS', 'TEMP_MIN_ACC_MENS', 'TE...",0.37573
816,"('TEMP_MÉD_MIN_MENS', 'TEMP_MÉD_MAX_MENS', 'TE...",0.37592
911,"('TEMP_MIN_MÉD_MENS', 'TEMP_MÉD_MIN_MENS', 'TE...",0.37688
114,"('TEMP_MÉD_MÉD_MENS', 'TEMP_MÉD_ACC_MENS', 'TE...",0.37718


In [12]:
fitness_regressao = fitness_regressao.sort_values("RRMSE").head(1).reset_index(drop=True)
fitness_regressao = pd.DataFrame(
    str(fitness_regressao.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "),
    columns=["FEATURES"])

fitness_regressao

,FEATURES
0,TEMP_MÉD_ACC_MENS
1,TEMP_MIN_MAX_MENS
2,TEMP_MIN_ACC_MENS
3,PRECIPITAÇÃO_MIN_MENS
4,TEMP_MAX_ACC_MENS
5,PRECIPITAÇÃO_MAX_MENS
6,DIA_DA_SEMANA_sex
7,MÊS_abr
8,MÊS_jul
9,MÊS_mai
